# Tutorial: py-statial on Keren et al. 2018 MIBI-TOF Data

This notebook walks through the py-statial API using the Keren et al. 2018 breast cancer MIBI-TOF dataset.

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import statial
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print(f'py-statial version: {statial.__version__}')

## 1. Load data

Load the cell metadata from the Keren et al. dataset. Each cell has spatial coordinates (x, y), a cell type, and an image ID.

In [ ]:
meta = pd.read_csv('../data/cell_metadata.csv', index_col=0)
print(f'Cells: {len(meta)}')
print(f'Images: {meta["imageID"].nunique()}')
print(f'Cell types: {meta["cellType"].nunique()}')
print(f'\nCell types: {sorted(meta["cellType"].unique())}')
meta.head()

## 2. get_distances — Pairwise cell type distances

In [ ]:
np.random.seed(42)
adata = ad.AnnData(X=np.random.randn(len(meta), 10), obs=meta)
adata = statial.get_distances(adata, max_dist=200, spatial_coords=['x', 'y'])

print(f'Distance matrix shape: {adata.obsm["distances"].shape}')
print(f'Cell types: {adata.uns["distances_columns"]}')

## 3. get_abundances — Cell type abundances

In [ ]:
adata = statial.get_abundances(adata, r=200, spatial_coords=['x', 'y'])
print(f'Abundance matrix shape: {adata.obsm["abundances"].shape}')

## 4. Kontextual — Conditional spatial relationships

In [ ]:
result = statial.Kontextual(
    cells=meta, r=50,
    from_types='Macrophages', to_types='Keratin_Tumour',
    parent=['Macrophages', 'CD4_Cell'],
    image=['6'],
    spatial_coords=['x', 'y'],
)
print('Kontextual result:')
result

## 5. kontext_curve — Kontextual over a range of radii

In [ ]:
rs = statial.kontext_curve(
    cells=meta,
    from_types='Macrophages', to_types='Keratin_Tumour',
    parent=['Macrophages', 'CD4_Cell'],
    rs=np.arange(10, 210, 50),
    image=['6'],
    spatial_coords=['x', 'y'],
)
print('Kontextual curve:')
rs

## 6. parent_combinations — Generate all pairwise relationships

In [ ]:
combos = statial.parent_combinations(
    all_types=['Macrophages', 'CD4_Cell', 'Keratin_Tumour', 'Tumour'],
    immune=['Macrophages', 'CD4_Cell'],
    tumour=['Keratin_Tumour', 'Tumour'],
)
print(f'{len(combos)} combinations generated:')
combos.head(10)

## 7. make_window — Create observation windows

In [ ]:
img6 = meta[meta['imageID'].astype(str) == '6']

w_square = statial.make_window(img6, window='square')
w_convex = statial.make_window(img6, window='convex')

print(f'Square window: {w_square["type"]}, xrange={w_square["xrange"]}')
print(f'Convex window: {w_convex["type"]}, polygon vertices={len(w_convex["polygon"])}')

## 8. prep_matrix — Convert results to matrix format

In [ ]:
if len(result) > 0:
    mat = statial.prep_matrix(result)
    print(f'Matrix shape: {mat.shape}')
    mat

## 9. relabel — Permutation testing

In [ ]:
relabel_df = statial.relabel(img6, labels=['Macrophages', 'CD4_Cell'], seed=42)
print(f'Original Macrophages: {(img6["cellType"] == "Macrophages").sum()}')
print(f'Relabeled Macrophages: {(relabel_df["cellType"] == "Macrophages").sum()}')
print('Label counts preserved (permuted within parent population)')

## 10. Validation

In [ ]:
print(f'is_kontextual: {statial.is_kontextual(result)}')
print(f'is_kontextual (bad): {statial.is_kontextual(pd.DataFrame({"x": [1]}))}')